Cell 1: Imports and Styling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Styling for dark mode (optional, based on your previous notebook)
plt.style.use('dark_background')
COLORS = ['#60a5fa','#f87171','#4ade80','#fb923c','#c084fc','#facc15','#34d399','#f472b6']

Cell 2: Data Loading and Label Consolidation

In [ ]:
df_raw = pd.read_csv('../perf_metrics.csv')

# 1. Consolidate 'Normal' labels to create a stronger baseline
normal_labels = ['normal_idle', 'normal_light', 'normal_mixed']
df_raw['label'] = df_raw['label'].replace(normal_labels, 'normal')

# 2. Derived basic columns
df_raw['total_faults'] = df_raw.get('minor_faults', 0) + df_raw.get('kernel_faults', 0)
df_raw['total_lock_contentions'] = (
    df_raw.get('mutex_contentions', 0) +
    df_raw.get('rwsem_read_contentions', 0) +
    df_raw.get('rwsem_write_contentions', 0)
)
df_raw['involuntary_pct'] = np.where(
    df_raw['ctx_switches'] > 0,
    df_raw['involuntary_switches'] / df_raw['ctx_switches'] * 100, 0
)

print(f"Dataset loaded: {len(df_raw)} rows")
print("Value counts:\n", df_raw['label'].value_counts())

Cell 3: Advanced Cleaning and Rate-Based Engineering

In [ ]:
# 1. Get only the numeric columns from your cleaned data
numeric_df = df_clean.select_dtypes(include=[np.number])

# 2. Calculate correlation for EVERYTHING numeric
full_corr = numeric_df.corr()

# 3. Plot it (we'll make the figure bigger so you can actually read the labels)
plt.figure(figsize=(20, 16))
sns.heatmap(full_corr, annot=False, cmap='coolwarm', center=0) # Set annot=False to keep it clean
plt.title("Full Dataset Correlation Matrix (Numeric Only)")
plt.show()

# 4. Find anything that correlates highly with 'involuntary_pct' as an example
print("Correlations with Involuntary %:")
print(full_corr['involuntary_pct'].sort_values(ascending=False).head(10))

In [ ]:
import numpy as np

# 1. Sparsity filter (Ensures we only keep rows with actual activity)
activity_cols = ['ctx_switches', 'minor_faults', 'total_alloc_bytes', 'futex_count']
df_clean = df_raw[(df_raw[activity_cols] > 0).any(axis=1)].copy()

# 2. Prevent division by zero: clip the runtime to a tiny positive value
# This ensures that even if runtime is 0, we don't get 'inf'
runtime_ms = (df_clean['total_runtime_ns'] / 1_000_000).clip(lower=1e-6)

# 3. Feature Engineering: Convert counts to rates
rate_features = ['ctx_switches', 'read_bytes', 'write_bytes', 'syscall_count', 'total_faults']
for col in rate_features:
    df_clean[f'{col}_rate'] = df_clean[col] / runtime_ms

# 4. Define final feature list
features = [
    'involuntary_pct', 'avg_runq_latency_ns', 'avg_syscall_latency_ns',
    'ctx_switches_rate', 'read_bytes_rate', 'write_bytes_rate', 
    'syscall_count_rate', 'total_faults_rate', 'total_lock_contentions'
]

# 5. Final Safety Clean: Handle any stray inf or NaN values
X = df_clean[features].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Final check for finiteness
if not np.isfinite(X).all().all():
    print("⚠️ Warning: Non-finite values still detected. Clipping extreme values.")
    X = X.clip(lower=-1e12, upper=1e12)

y = df_clean['label']
le = LabelEncoder()
y_enc = le.fit_transform(y)

# 6. Train/Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

print(f"Data ready for training. X shape: {X_train.shape}")

Cell 4: The Evaluation Function

In [ ]:
def run_model_test(name, model, X_t, y_t, X_v, y_v):
    print(f"\n--- Testing {name} ---")
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    
    print(classification_report(y_v, preds, target_names=le.classes_))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_v, preds, normalize='true')
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

Cell 5: Compare Models (with cuda)

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    # Added 'tree_method' for your GPU
    "XGBoost (GPU)": XGBClassifier(device="cuda", n_estimators=100, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

for name, model in models.items():
    run_model_test(name, model, X_train, y_train, X_test, y_test)